# Week 1 Discovery

In [1]:
# load libraries 
import pandas as pd 
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import numpy as np
import pyarrow.parquet as pypq
import textwrap 
from time import time 
import plotly.io as pio

tqdm.pandas()
plt.rcParams.update({'font.size': 22})
sns.set(style="ticks", context="talk")
plt.style.use("dark_background")
pd.options.plotting.backend = 'plotly'
pio.templates.default = 'plotly_dark+presentation'

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [2]:
# some helper functions 
def read_parquet(path, engine='pyarrow', columns=None, convert_dtypes=True, **args):
    """
    Read a parquet file (or a directory of parquet files) 
    columns: list of columns to read, by default, read all columns
    convert_dtypes: if True, convert datatypes to save RAM (takes extra time)
    """
    name = path.stem 
    column_st = 'columns="all"' if columns is None else f'{columns=!r}'
    print(f'\nReading {column_st} from {path!r} using {engine=!r}.')

    tic = time()
    df = pd.read_parquet(path, engine=engine, columns=columns, **args)
    toc = time()
    print(f'Read {len(df):,} rows from {path.stem!r} in {toc-tic:.2f} sec.')
    
    if convert_dtypes:
        tic = time()
        size_before = df.memory_usage(deep=True).sum() / 1024 / 1024 / 1024

        string_cols_d = {}
        for col, dtype in df.dtypes.to_dict().items():
            if dtype == 'object':  # convert object columns to string
                string_cols_d[col] = 'string[python]'
            if col == 'type' or col == 'concept_name':
                if dtype != 'category':
                    string_cols_d[col] = 'category'
            if col == 'publication_month':
                if dtype != 'uint8':
                    string_cols_d[col] = 'uint8'
            if col == 'score':
                if dtype != 'float16':
                    string_cols_d[col] = 'float16'
        # print(f'{string_cols_d=}')
        df = df.astype(string_cols_d) 
        
        size_after = df.memory_usage(deep=True).sum() / 1024 / 1024 / 1024
        toc = time()
        print(f'Converting dtypes took {toc-tic:.2f} sec. Size before: {size_before:.2f}GB, after: {size_after:.2f}GB.')
    
    display('Top 3 rows:', df.head(3))
    return df


def peek_parquet(path):
    """
    peeks at a parquet file (or a directory containing parquet files) without reading the whole thing and prints the following:
    * Path
    * schema
    * number of pieces (fragments)
    * number of rows 
    """
    parq_file = pypq.ParquetDataset(path)
    piece_count = len(parq_file.fragments)
    schema = textwrap.indent(parq_file.schema.to_string(), ' '*4)
    row_count = sum(frag.count_rows() for frag in parq_file.fragments)
    if Path(path).is_dir():
      size = sum(Path(frag.path).stat().st_size for frag in parq_file.fragments)
    else:
      size = path.stat().st_size
    
    st = [
        f'Name: {path.stem!r}',  
        f'Path: {str(path)!r}',
        f'Size: {size/1024/1024/1024:.2g} GB',
        f'Files: {piece_count:,}',
        f'Rows: {row_count:,}',
        f'Schema:\n{schema}',
        f'5 random rows:',
    ]
    print('\n'.join(st))
    sample_df = parq_file.fragments[0].head(5).to_pandas()  # read 5 rows from the first fragment
    display(sample_df)

    return

def read_smaller_tables(name):
    """
    Some smaller tables exist as a CSV only
    """
    assert name in ['institutions', 'institutions_geo', 'concepts']
    path = basepath / 'csv-files'/ month / name
    df = pd.read_csv(f'{path}.csv.gz', engine='c')
    return df

def tsv_to_parquet(tsv_path, output_filename=None, dtype=None, **read_csv_kwargs):
    """
    Convert a TSV file to Parquet format and save it in the current directory.
     
    Args:
        tsv_path: Path to the TSV file to convert
        output_filename: Optional output filename (defaults to same name as TSV but .parquet)
        dtype: Optional dict of column dtypes for reading TSV
        **read_csv_kwargs: Additional arguments to pass to pd.read_csv
    
    Returns:
        Path to the created parquet file
    """

    #get the output path
    notebook_dir = Path.cwd()
    if output_filename is None:
        output_filename = Path(tsv_path).stem + '.parquet'
    output_path = notebook_dir / output_filename

    if not output_path.exists():
        print(f"Converting {tsv_path} to Parquet...")
        df = pd.read_csv(
            tsv_path,
            sep = '\t',
            dtype=dtype,
            low_memory=False, 
            **read_csv_kwargs
        )
        df.to_parquet(output_path, index=False)

        print(f"Successfully wrote Parquet file to: {output_path}")

    return output_path         

In [3]:
from pathlib import Path
print(Path('/data/shared/USPTO-patents/g_patent.tsv').stat().st_size / 1024**3)
print(Path('/data/shared/USPTO-patents/g_us_patent_citation.tsv').stat().st_size / 1024**3)
print(Path('/data/shared/USPTO-patents/g_inventor_disambiguated.tsv').stat().st_size / 1024**3)
print(Path('/data/shared/USPTO-patents/g_us_application_citation.tsv').stat().st_size / 1024**3)
print(Path('/data/shared/USPTO-patents/g_wipo_technology.tsv').stat().st_size / 1024**3)
print(Path('/data/shared/USPTO-patents/g_foreign_citation.tsv').stat().st_size / 1024**3)
print(Path('/data/shared/USPTO-patents/g_inventor_disambiguated.tsv').stat().st_size /1024**3)
print(Path('/data/shared/USPTO-patents/g_uspc_at_issue.tsv').stat().st_size /1024**3)
print(Path('g_uspc_at_issue.parquet').stat().st_size /1024**3)
print(Path('g_patent.parquet').stat().st_size / 1024**3)

#so far tsvs <2GB can be converted straight up


1.0351925855502486
10.013551264069974
2.1136436695232987
5.693413279019296
0.7282795011997223
2.605386941693723
2.1136436695232987
1.9600079851225019
0.6690510278567672
0.3170516174286604


In [4]:
# Convert TSV to Parquet (only runs if parquet doesn't exist yet)
from pathlib import Path
import pandas as pd

def tsv_to_parquet(tsv_path, output_filename=None, dtype=None, **read_csv_kwargs):
    """
    Convert a TSV file to Parquet format and save it in the current directory.
     
    Args:
        tsv_path: Path to the TSV file to convert
        output_filename: Optional output filename (defaults to same name as TSV but .parquet)
        dtype: Optional dict of column dtypes for reading TSV
        **read_csv_kwargs: Additional arguments to pass to pd.read_csv
    
    Returns:
        Path to the created parquet file
    """

    #get the output path
    notebook_dir = Path.cwd()
    if output_filename is None:
        output_filename = Path(tsv_path).stem + '.parquet'
    output_path = notebook_dir / output_filename

    if not output_path.exists():
        print(f"Converting {tsv_path} to Parquet...")
        df = pd.read_csv(
            tsv_path,
            sep = '\t',
            dtype=dtype,
            low_memory=False, 
            **read_csv_kwargs
        )
        df.to_parquet(output_path, index=False)

        print(f"Successfully wrote Parquet file to: {output_path}")

    return output_path         

Patent_output_path = tsv_to_parquet('/data/shared/USPTO-patents/g_patent.tsv', dtype={'patent_id': str})

wipo_output_path = tsv_to_parquet('/data/shared/USPTO-patents/g_wipo_technology.tsv', dtype={'patent_id': str, 'wipo_sector_title': str, 'wipo_field_title': str})

## does not work even with dtypes still too big to perform application_output_path = tsv_to_parquet('/data/shared/USPTO-patents/g_us_application_citation.tsv', dtype={'patent_id': str, 'citation_document_number': str, 'record_name': str, 'wipo_kind': str, 'citation_category': str,})




##df = pd.read_csv('/data/shared/USPTO-patents/g_patent.tsv', sep = '\t')
df = pd.read_parquet('g_patent.parquet')

df.iloc[345]

patent_id                                                10000349
patent_type                                               utility
patent_date                                            2018-06-19
patent_title    Drive transmission device and sheet conveying ...
wipo_kind                                                      B2
num_claims                                                      7
withdrawn                                                       0
filename                                            ipg180619.xml
Name: 345, dtype: object

In [ ]:
import pandas as pd

# Load the parquet file
#wipo_df = pd.read_parquet('g_wipo_technology.parquet')
uspc_at_issue_df = pd.read_parquet('g_uspc_at_issue.parquet')
g_patent_df = pd.read_parquet('g_patent.parquet')

##print(uspc_at_issue_df.head())
print(uspc_at_issue_df.iloc[6])




In [4]:

import pandas as pd
df = pd.read_csv('/data/shared/USPTO-patents/g_cpc_title.tsv', sep = '\t')
df.head(2000)

,cpc_subclass,cpc_subclass_title,cpc_group,cpc_group_title,cpc_class,cpc_class_title
0,A01B,SOIL WORKING IN AGRICULTURE OR FORESTRY; PARTS...,A01B1/00,Hand tools,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
1,A01B,SOIL WORKING IN AGRICULTURE OR FORESTRY; PARTS...,A01B1/02,Hand tools -Spades; Shovels,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
2,A01B,SOIL WORKING IN AGRICULTURE OR FORESTRY; PARTS...,A01B1/022,Hand tools -Spades; Shovels -Collapsible; exte...,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
3,A01B,SOIL WORKING IN AGRICULTURE OR FORESTRY; PARTS...,A01B1/024,Hand tools -Spades; Shovels -Foot protectors a...,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
4,A01B,SOIL WORKING IN AGRICULTURE OR FORESTRY; PARTS...,A01B1/026,Hand tools -Spades; Shovels -with auxiliary ha...,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
...,...,...,...,...,...,...
1995,A01K,ANIMAL HUSBANDRY; AVICULTURE; APICULTURE; PISC...,A01K2207/20,Modified animals-Animals treated with compound...,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
1996,A01K,ANIMAL HUSBANDRY; AVICULTURE; APICULTURE; PISC...,A01K2207/25,Modified animals-Animals on a special diet,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
1997,A01K,ANIMAL HUSBANDRY; AVICULTURE; APICULTURE; PISC...,A01K2207/30,Modified animals-Animals modified by surgical ...,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...
1998,A01K,ANIMAL HUSBANDRY; AVICULTURE; APICULTURE; PISC...,A01K2207/35,Modified animals-Animals modified by environme...,A01,AGRICULTURE; FORESTRY; ANIMAL HUSBANDRY; HUNTI...


In [1]:
import pandas as pd
USPTOPath = '/data/shared/USPTO-patents/'
df = pd.read_csv(USPTOPath+'g_cpc_current.tsv', sep = '\t')
df.head(5)

,patent_id,cpc_sequence,cpc_section,cpc_class,cpc_subclass,cpc_group,cpc_type
0,3950000,0,A,A63,A63C,A63C9/001,inventional
1,3950000,1,A,A63,A63C,A63C9/00,inventional
2,3950000,2,A,A63,A63C,A63C9/002,inventional
3,3950000,3,A,A63,A63C,A63C9/081,inventional
4,3950001,0,A,A63,A63C,A63C9/086,inventional
